### Course Popularity - Final Project for CSC440: Data Mining


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc

from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

# ============================================================
# 0. Load CSVs
# ============================================================
sections_file_path = Path("Data") / "Course_data" / "courses.sections.csv"
sectionsTS_file_path = Path("Data") / "Course_data" / "courses.sectionsTS.csv"

sections_df = pd.read_csv(sections_file_path)
sections_ts_df = pd.read_csv(sectionsTS_file_path)

print("\n========== Loaded Datasets ==========")
print("courses.sections.csv info:")
print(sections_df.info(), "\n")

print("courses.sectionsTS.csv info:")
print(sections_ts_df.info(), "\n")

print("Static sections columns:", sections_df.columns)
print("Time-series columns:", sections_ts_df.columns)

/var/folders/k3/56x6m5_17v3d1lgq1hgllvhm0000gn/T/ipykernel_91822/3052413460.py:11: DtypeWarning: Columns (17,18,29,30,31,32,33,34,35,46,48,49,50,51,52,53,54,57,58,59,60,61,62,63,64,65,66,67,68,69,75,76,77,78,84,85,86,87,93,94,95,96,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,142,143,144,145,151,152,153,154,157,158,159,160,161) have mixed types. Specify dtype option on import or set low_memory=False.
  sections_df = pd.read_csv(sections_file_path)


courses.sections.csv info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29441 entries, 0 to 29440
Columns: 162 entries, _id to topic[4]
dtypes: bool(1), float64(2), int64(4), object(155)
memory usage: 36.2+ MB
None 

courses.sections.csv:
                        _id                                name  \
0  67c083cef212c01be3209013  Principles of Financial Accounting   
1  67c083cef212c01be3209014  Principles of Financial Accounting   
2  67c083cef212c01be3209015  Principles of Financial Accounting   
3  67c083cef212c01be3209016  Principles of Financial Accounting   
4  67c083cef212c01be3209017  Principles of Financial Accounting   

            subjectName subjectCode catalogNumber academicCareer semester  \
0  Accounting Bus Admin         ACC           211  Undergraduate   Spring   
1  Accounting Bus Admin         ACC           211  Undergraduate   Spring   
2  Accounting Bus Admin         ACC           211  Undergraduate   Spring   
3  Accounting Bus Admin         ACC          

In [ ]:
# ============================================================
# 1. Helper: Build Data Quality Report for a DataFrame
# ============================================================

def build_dq_report(df: pd.DataFrame, df_name: str):
    """
    Perform data quality checks and print all results:
    - Overview
    - Null summary
    - Dtype summary
    - Numeric summary
    - Categorical summary
    - Type coercion results
    """
    print("\n\n============================================================")
    print(f"############  DATA QUALITY REPORT: {df_name}  ############")
    print("============================================================\n")

    # ---------- Overview ----------
    print("---- Overview ----")
    overview = pd.DataFrame({
        "n_rows": [df.shape[0]],
        "n_columns": [df.shape[1]],
        "n_duplicates": [df.duplicated().sum()]
    })
    print(overview, "\n")

    # ---------- Null Summary ----------
    print("---- Null Summary (Top 20 sorted by % null) ----")
    null_summary = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.values.astype(str),
        "null_count": df.isna().sum().values,
        "null_pct": (df.isna().mean() * 100).values
    }).sort_values("null_pct", ascending=False)

    print(null_summary.head(20), "\n")

    # ---------- Dtype Summary ----------
    print("---- Dtype Summary ----")
    dtype_summary = (
        null_summary[["column", "dtype"]]
        .groupby("dtype")
        .agg(n_columns=("column", "count"))
        .reset_index()
    )
    print(dtype_summary, "\n")

    # ---------- Numeric Summary ----------
    print("---- Numeric Summary ----")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        numeric_summary = df[numeric_cols].describe().T
        numeric_summary["missing_count"] = df[numeric_cols].isna().sum()
        numeric_summary["missing_pct"] = df[numeric_cols].isna().mean() * 100
        print(numeric_summary, "\n")
    else:
        print("No numeric columns.\n")

    # ---------- Categorical Summary ----------
    print("---- Categorical Summary (Top 20 columns) ----")
    cat_cols = df.select_dtypes(include=["object", "category"]).columns
    cat_summary_rows = []
    for col in cat_cols:
        series = df[col]
        cat_summary_rows.append({
            "column": col,
            "dtype": str(series.dtype),
            "n_unique": series.nunique(dropna=True),
            "missing_pct": series.isna().mean() * 100,
            "top_5_values": ", ".join(
                [f"{idx}({val})"
                 for idx, val in series.value_counts(dropna=True).head(5).items()]
            )
        })
    if cat_summary_rows:
        cat_summary = pd.DataFrame(cat_summary_rows)
        print(cat_summary.head(20), "\n")
    else:
        print("No categorical columns.\n")

    # ---------- Type Coercion Checks ----------
    print("---- Type Coercion Checks ----")

    expected_numeric_cols = [
        "capacity", "waitlistCapacity", "reservedSeatsAvailable",
        "reservedSeatsCapacity", "year", "catalogNumber", "classNumber"
    ]
    expected_datetime_cols = [
        "dateTimeRetrieved", "timeStart", "timeEnd",
        "startDate", "endDate"
    ]

    # Numeric Type Coercion
    print("\nNumeric Columns Coercion Results:")
    for col in expected_numeric_cols:
        if col in df.columns:
            before = df[col].notna().sum()
            coerced = pd.to_numeric(df[col], errors="coerce")
            after = coerced.notna().sum()

            print(
                f"{col}: non-null before={before}, after={after}, "
                f"invalid converted={before - after}"
            )

            df[col] = coerced

    # Datetime Type Coercion
    print("\nDatetime Columns Coercion Results:")
    for col in expected_datetime_cols:
        if col in df.columns:
            before = df[col].notna().sum()
            coerced = pd.to_datetime(df[col], errors="coerce")
            after = coerced.notna().sum()

            print(
                f"{col}: non-null before={before}, after={after}, "
                f"invalid converted={before - after}"
            )

            df[col] = coerced

    print("\n========== END OF REPORT FOR:", df_name, "==========\n")

    return df


# ============================================================
# 2. Run Reports for Both Datasets
# ============================================================

sections_df = build_dq_report(sections_df.copy(), "courses.sections")
sections_ts_df = build_dq_report(sections_ts_df.copy(), "courses.sectionsTS")

gc.collect()

print("\nData quality analysis completed.\n")


In [ ]:
# ===============================================================
# 2. Merge Static Course Metadata with Time-Series Observations
# ===============================================================

# Merge on classNumber (primary key)
df = sections_ts_df.merge(
    sections_df,
    on="classNumber",
    how="left",
    suffixes=("", "_static")
)

# Convert timestamps
df["dateTimeRetrieved"] = pd.to_datetime(df["dateTimeRetrieved"], errors="coerce")
df = df.sort_values(["classNumber", "dateTimeRetrieved"])

# ===============================================================
# 3. Feature Engineering
# ===============================================================

# ---- Temporal Features ----
df["days_since_start"] = df.groupby("classNumber")["dateTimeRetrieved"] \
                           .transform(lambda x: (x - x.min()).dt.total_seconds() / 86400)

df["hour"] = df["dateTimeRetrieved"].dt.hour
df["day_of_week"] = df["dateTimeRetrieved"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)

# Cyclical time encoding
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

# ---- Enrollment Features ----
df["seats_taken"] = df["capacity"] - df["seatsAvailable"]
df["fill_pct"] = df["seats_taken"] / df["capacity"].replace(0, np.nan)

df["waitlist_ratio"] = df["waitlistAvailable"] / df["waitlistCapacity"].replace(0, np.nan)

# ---- Velocity Metrics ----
df["fill_velocity_per_day"] = df.groupby("classNumber")["fill_pct"].diff() / \
                              df.groupby("classNumber")["days_since_start"].diff()

df["rolling_velocity_6h"] = df.groupby("classNumber")["fill_pct"].transform(
    lambda x: x.diff().rolling(6).mean()
)

# ---- Course-Level Features ----
df["course_level"] = df["catalogNumber"] // 100
df["is_online"] = (df["classroom"] == "Online").astype(int)

df["timeStart"] = pd.to_datetime(df["timeStart"], errors="coerce")
df["early_morning"] = (df["timeStart"].dt.hour < 10).astype(int)
df["evening"] = (df["timeStart"].dt.hour >= 17).astype(int)

gc.collect()


# ===============================================================
# 4. Modeling Dataset Construction
# ===============================================================
target = "fill_pct"

feature_cols = [
    "days_since_start", "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "is_weekend", "seats_taken", "capacity", "waitlist_ratio",
    "fill_velocity_per_day", "rolling_velocity_6h",
    "course_level", "is_online", "early_morning", "evening"
]

# One-hot encode subjectCode & academicCareer
df = pd.get_dummies(df, columns=["subjectCode", "academicCareer"], drop_first=True)

feature_cols.extend([col for col in df.columns if col.startswith("subjectCode_")])
feature_cols.extend([col for col in df.columns if col.startswith("academicCareer_")])

df = df.dropna(subset=[target])

X = df[feature_cols]
y = df[target]

# ===============================================================
# 5. Time-Series Cross Validation (No Leakage)
# ===============================================================
tscv = TimeSeriesSplit(n_splits=5)

mae_scores, rmse_scores, r2_scores = [], [], []

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist"
)

for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae_scores.append(mean_absolute_error(y_test, preds))
    rmse_scores.append(mean_squared_error(y_test, preds, squared=False))
    r2_scores.append(r2_score(y_test, preds))


print("\n========== Time-Series CV Results ==========")
print("MAE:", np.mean(mae_scores))
print("RMSE:", np.mean(rmse_scores))
print("R²:", np.mean(r2_scores))


# ===============================================================
# 6. Business-Relevant Metric: ±5% Tolerance Accuracy
# ===============================================================
model.fit(X, y)
preds = model.predict(X)

tolerance = 0.05
within_tol = np.mean(np.abs(preds - y) <= tolerance)

print("\nWithin ±5% tolerance accuracy:", within_tol)


# ===============================================================
# 7. Popularity Metrics for Ranking Courses
# ===============================================================
course_groups = df.groupby("classNumber")

popularity = pd.DataFrame({
    "classNumber": course_groups["classNumber"].first(),
    "subjectCode": course_groups["subjectCode"].first(),
    "catalogNumber": course_groups["catalogNumber"].first(),
    "course_level": course_groups["course_level"].first(),
    
    # Popularity Scores:
    "final_fill_rate": course_groups["fill_pct"].last(),
    "fill_velocity_score": course_groups["fill_velocity_per_day"].mean(),
    "early_rush_score": course_groups["fill_pct"].nth(3),  # first few hours
    "waitlist_demand_score": course_groups["waitlist_ratio"].max()
})

# Composite Score
popularity["popularity_score"] = (
    0.40 * popularity["final_fill_rate"] +
    0.30 * popularity["fill_velocity_score"] +
    0.20 * popularity["waitlist_demand_score"] +
    0.10 * popularity["early_rush_score"]
)

# ===============================================================
# 8. Ranking Courses (CSC, ECE by 200/300/400 level)
# ===============================================================
target_subjects = ["CSC", "ECE"]
filtered = popularity[popularity["subjectCode"].isin(target_subjects)]

ranked_output = {}

for subj in target_subjects:
    subj_df = filtered[filtered["subjectCode"] == subj]
    ranked_output[subj] = {}

    for level in [200, 300, 400]:
        lvl = subj_df[subj_df["course_level"] == level]
        ranked_output[subj][level] = lvl.sort_values(
            "popularity_score", ascending=False
        ).head(10)  # top 10

# Print Sample
print("\n=========== Top CSC 200-Level Courses ===========")
print(ranked_output["CSC"][200].head())

print("\n=========== Top ECE 300-Level Courses ===========")
print(ranked_output["ECE"][300].head())

# Optionally export results
popularity.to_csv("course_popularity_scores.csv", index=False)
